In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"
import numpy as np
import pandas as pd
import joblib
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import recall_score, f1_score, classification_report, confusion_matrix, precision_score

# 재현성 고정
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

### 1. data load

In [2]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=["target"])
y_train = train["target"]

X_val = val.drop(columns=["target"])
y_val = val["target"]

X_test = test.drop(columns=["target"])
y_test = test["target"]

print(X_train.shape, X_val.shape, X_test.shape)  # (2654, 81) (885, 81) (885, 81)

(2654, 81) (885, 81) (885, 81)


### 2. MLP Model 구성

In [3]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Recall(name="recall")]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,761 (85.00 KB)

 Trainable params: 21,313 (83.25 KB)

 Non-trainable params: 448 (1.75 KB)

### 3-1. Threshold = 0.50 (기본값) 학습

In [ ]:
import tensorflow.keras.backend as K

def f1_metric(y_true, y_pred):
    y_true = K.cast(y_true, "float32")      
    y_pred = K.cast(K.round(y_pred), "float32")
    
    tp = K.sum(y_true * y_pred)
    fp = K.sum((1 - y_true) * y_pred)
    fn = K.sum(y_true * (1 - y_pred))
    
    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    return 2 * precision * recall / (precision + recall + K.epsilon())

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.Recall(name="recall"), f1_metric]
)

early_stop = keras.callbacks.EarlyStopping( 
    monitor="val_f1_metric",
    mode="max",
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
83/83 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8674 - f1_metric: 0.7975 - loss: 0.3250 - recall: 0.8556 - val_accuracy: 0.8655 - val_f1_metric: 0.7822 - val_loss: 0.3330 - val_recall: 0.8175
Epoch 2/200
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8625 - f1_metric: 0.7906 - loss: 0.3313 - recall: 0.8439 - val_accuracy: 0.8644 - val_f1_metric: 0.7864 - val_loss: 0.3386 - val_recall: 0.8316
Epoch 3/200
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8704 - f1_metric: 0.8029 - loss: 0.3148 - recall: 0.8580 - val_accuracy: 0.8599 - val_f1_metric: 0.7779 - val_loss: 0.3474 - val_recall: 0.8246
Epoch 4/200
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8760 - f1_metric: 0.8098 - loss: 0.3032 - recall: 0.8662 - val_accuracy: 0.8734 - val_f1_metric: 0.7975 - val_loss: 0.3385 - val_recall: 0.8351
Epoch 5/200
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8772 - f1_metric: 0.8121 - loss: 0.2939 - recall: 0.8732 - val_accuracy: 0.8757 - val_f1_metric: 0.

#### 3-1. Threshold = 0.50 (기본값) 평가

In [10]:
val_proba = model.predict(X_val).flatten()
val_pred = (val_proba >= 0.5).astype(int)

print(f"Recall: {recall_score(y_val, val_pred):.4f}")
print(f"F1: {f1_score(y_val, val_pred):.4f}")
print(classification_report(y_val, val_pred))

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Recall: 0.8351
F1: 0.8151
              precision    recall  f1-score   support

           0       0.92      0.90      0.91       600
           1       0.80      0.84      0.82       285

    accuracy                           0.88       885
   macro avg       0.86      0.87      0.86       885
weighted avg       0.88      0.88      0.88       885



### 3-2. Threshold = 0.40 + 평가

In [15]:
val_proba = model.predict(X_val).flatten()
val_pred = (val_proba >= 0.40).astype(int)

print(f"Recall: {recall_score(y_val, val_pred):.4f}")
print(f"F1: {f1_score(y_val, val_pred):.4f}")
print(classification_report(y_val, val_pred))

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Recall: 0.8421
F1: 0.8040
              precision    recall  f1-score   support

           0       0.92      0.88      0.90       600
           1       0.77      0.84      0.80       285

    accuracy                           0.87       885
   macro avg       0.85      0.86      0.85       885
weighted avg       0.87      0.87      0.87       885



### 4. Model Save 

In [16]:
import json

FINAL_THRESHOLD = 0.40

val_pred = (val_proba >= FINAL_THRESHOLD).astype(int)
print(f"최종 채택 threshold: {FINAL_THRESHOLD}")
print(f"Recall: {recall_score(y_val, val_pred):.4f}")
print(f"Precision: {precision_score(y_val, val_pred):.4f}")
print(f"F1: {f1_score(y_val, val_pred):.4f}")

# 모델 저장
model.save("../models/mlp.keras")

# threshold 별도 저장 (Streamlit 앱에서 그대로 재사용)
with open("../models/mlp_threshold.json", "w") as f:
    json.dump({"threshold": FINAL_THRESHOLD}, f)

print("저장 완료: models/mlp.keras, models/mlp_threshold.json")

최종 채택 threshold: 0.4
Recall: 0.8421
Precision: 0.7692
F1: 0.8040
저장 완료: models/mlp.keras, models/mlp_threshold.json
